In [ ]:
%pip install catboost scikit-learn rdkit seaborn optuna

In [ ]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn import set_config
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, DataStructs
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

## Phase 1 — Descriptor computation
Run once. Results cached to `descriptors.csv`.

In [ ]:
# Run notebook from its own directory, or adjust path as needed
raw = pd.read_csv("physical_chemical_properties_of_organic_substances.csv")
print(f"Raw dataset: {raw.shape[0]} rows, {raw.shape[1]} columns")
raw.head()

In [ ]:
# Keep only organic molecules — drop inorganics and salts (dot-notation SMILES)
properties = raw[['name', 'smiles']].copy()
print(f"Before filtering: {properties.shape[0]}")
properties = properties[properties['smiles'].notna()]
properties = properties[~properties['smiles'].str.contains('.', regex=False, na=False)]
carbon_regex = r'c|[C](?![a-z])'
properties = properties[properties['smiles'].str.contains(carbon_regex, regex=True, na=False)]
print(f"After organic filter: {properties.shape[0]}")

In [ ]:
# RDKit 2D descriptors — parse each SMILES once
def safe_descriptors(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    return Descriptors.CalcMolDescriptors(mol)

desc_series = properties['smiles'].apply(safe_descriptors)
valid_mask = desc_series.notna()
descriptor_list = pd.DataFrame(desc_series[valid_mask].tolist(), index=properties.index[valid_mask])
properties = properties.loc[valid_mask]
properties = pd.concat([properties, descriptor_list], axis=1)
print(f"After 2D descriptors: {properties.shape}")

In [ ]:
# Morgan fingerprints (radius 2, 2048 bits)
fps_list, fps_indices = [], []
for index, smi in properties['smiles'].items():
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    arr = np.zeros((2048,), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    fps_list.append(arr)
    fps_indices.append(index)

fps = pd.DataFrame(fps_list, index=fps_indices, columns=[f'fp_{i}' for i in range(2048)])
properties = properties.loc[fps_indices]
properties = pd.concat([properties, fps], axis=1)
print(f"After fingerprints: {properties.shape}")

In [ ]:
TARGETS = [
    'melting_point_K', 'boiling_point_K', 'heat_of_fusion',
    'heat_of_vaporization', 'critical_temperature', 'critical_pressure', 'flash_point'
]
for t in TARGETS:
    properties[t] = raw[t]

properties = properties.loc[:, ~properties.columns.duplicated()]
print(f"Final feature matrix: {properties.shape}")
print("Valid rows per target:")
print(properties[TARGETS].notna().sum())

In [ ]:
properties.to_csv("descriptors.csv", index=False)
print("Saved descriptors.csv")

In [ ]:
properties = pd.read_csv("descriptors.csv", low_memory=False)
properties.head()

## Phase 2 — Training
Set `property_to_predict` below and run the remaining cells. Takes ~3–5 min per property.

In [ ]:
property_to_predict = 'melting_point_K'
# Options: melting_point_K, boiling_point_K, heat_of_fusion,
#          heat_of_vaporization, critical_temperature, critical_pressure, flash_point

subset = properties[properties[property_to_predict].notna()].copy()
Q1, Q3 = subset[property_to_predict].quantile([0.25, 0.75])
IQR = Q3 - Q1
subset = subset[
    (subset[property_to_predict] >= Q1 - 1.5 * IQR) &
    (subset[property_to_predict] <= Q3 + 1.5 * IQR)
]
print(f"{property_to_predict}: {len(subset)} rows after cleaning")

In [ ]:
set_config(transform_output="pandas")

X = subset.drop(columns=['name', 'smiles'] + TARGETS)
X = X.apply(pd.to_numeric, errors='coerce').dropna()
y = subset[property_to_predict][X.index]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train, X_val,  y_train, y_val  = train_test_split(X_train, y_train, test_size=0.15, random_state=42)

preprocessor = Pipeline(steps=[
    ('variance', VarianceThreshold(threshold=0.0)),
    ('scaler',   StandardScaler())
])
preprocessor.fit(X_train)
X_train_p = preprocessor.transform(X_train)
X_val_p   = preprocessor.transform(X_val)
X_test_p  = preprocessor.transform(X_test)

def objective(trial):
    params = {
        "iterations":    trial.suggest_int("iterations", 500, 3000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "depth":         trial.suggest_int("depth", 3, 6),
        "l2_leaf_reg":   trial.suggest_float("l2_leaf_reg", 1e-2, 100.0, log=True),
        "rsm":           trial.suggest_float("rsm", 0.1, 1.0),
        "loss_function": "RMSE",
        "early_stopping_rounds": 100,
        "verbose": False,
    }
    m = CatBoostRegressor(**params)
    m.fit(X_train_p, y_train, eval_set=(X_val_p, y_val), use_best_model=True)
    return np.sqrt(mean_squared_error(y_val, m.predict(X_val_p)))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)
print(f"Best trial RMSE: {study.best_value:.3f}")

best_params = study.best_params
best_params.update({"loss_function": "RMSE", "early_stopping_rounds": 100, "verbose": 200})
model = CatBoostRegressor(**best_params)
model.fit(X_train_p, y_train, eval_set=(X_val_p, y_val), use_best_model=True)

predictions = model.predict(X_test_p)
mask = y_test.abs() > 1e-8
mape = float(np.mean(np.abs((y_test[mask] - predictions[mask]) / y_test[mask])))
mae  = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2   = r2_score(y_test, predictions)

print(f"\nTest results for {property_to_predict}")
print(f"  RMSE: {rmse:.3f}")
print(f"  MAPE: {mape:.3f}")
print(f"  MAE:  {mae:.3f}")
print(f"  R\u00b2:   {r2:.3f}")

### Analysis

In [ ]:
error_analysis = pd.DataFrame(y_test).rename(columns={property_to_predict: 'actual'})
error_analysis['predicted'] = predictions
error_analysis['smiles'] = properties.loc[error_analysis.index, 'smiles']
error_analysis['error'] = abs(error_analysis['actual'] - error_analysis['predicted'])
error_analysis = error_analysis.sort_values(by='error', ascending=False)
error_analysis

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x='actual', y='predicted', data=error_analysis, alpha=0.6, edgecolor=None)
lims = [min(error_analysis['actual'].min(), error_analysis['predicted'].min()),
        max(error_analysis['actual'].max(), error_analysis['predicted'].max())]
plt.plot(lims, lims, color='red', linestyle='--', linewidth=2)
plt.title(f'Actual vs Predicted \u2014 {property_to_predict}  (R\u00b2={r2:.3f}, MAPE={mape:.3f})')
plt.xlabel(f'Actual {property_to_predict}')
plt.ylabel(f'Predicted {property_to_predict}')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
importances = model.get_feature_importance()
feature_names = X_train_p.columns
importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
importance_df = importance_df.sort_values('importance', ascending=False)
print(f"Top 20 features for {property_to_predict}:")
print(importance_df.head(20).to_string(index=False))

## All-targets summary
Trains all seven properties and prints a combined results table.

In [ ]:
all_results = {}

for target in TARGETS:
    subset = properties[properties[target].notna()].copy()
    Q1, Q3 = subset[target].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    subset = subset[(subset[target] >= Q1 - 1.5*IQR) & (subset[target] <= Q3 + 1.5*IQR)]

    X = subset.drop(columns=['name', 'smiles'] + TARGETS)
    X = X.apply(pd.to_numeric, errors='coerce').dropna()
    y = subset[target][X.index]

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
    X_tr, X_v,  y_tr, y_v  = train_test_split(X_tr, y_tr, test_size=0.15, random_state=42)

    pre = Pipeline(steps=[('var', VarianceThreshold(0.0)), ('scl', StandardScaler())])
    pre.fit(X_tr)
    X_tr_p = pre.transform(X_tr)
    X_v_p  = pre.transform(X_v)
    X_te_p = pre.transform(X_te)

    def obj(trial):
        p = {
            "iterations":    trial.suggest_int("iterations", 500, 3000),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "depth":         trial.suggest_int("depth", 3, 6),
            "l2_leaf_reg":   trial.suggest_float("l2_leaf_reg", 1e-2, 100.0, log=True),
            "rsm":           trial.suggest_float("rsm", 0.1, 1.0),
            "loss_function": "RMSE", "early_stopping_rounds": 100, "verbose": False,
        }
        m = CatBoostRegressor(**p)
        m.fit(X_tr_p, y_tr, eval_set=(X_v_p, y_v), use_best_model=True)
        return np.sqrt(mean_squared_error(y_v, m.predict(X_v_p)))

    st = optuna.create_study(direction="minimize")
    st.optimize(obj, n_trials=15)

    bp = st.best_params
    bp.update({"loss_function": "RMSE", "early_stopping_rounds": 100, "verbose": 0})
    m = CatBoostRegressor(**bp)
    m.fit(X_tr_p, y_tr, eval_set=(X_v_p, y_v), use_best_model=True)

    preds = m.predict(X_te_p)
    _mask = y_te.abs() > 1e-8
    _safe_mape = float(np.mean(np.abs((y_te[_mask] - preds[_mask]) / y_te[_mask])))
    all_results[target] = {
        'n':    len(subset),
        'RMSE': round(float(np.sqrt(mean_squared_error(y_te, preds))), 3),
        'MAPE': round(_safe_mape, 3),
        'MAE':  round(float(mean_absolute_error(y_te, preds)), 3),
        'R2':   round(float(r2_score(y_te, preds)), 3),
    }
    print(f"{target:30s}  n={len(subset):4d}  MAPE={all_results[target]['MAPE']:.3f}  R\u00b2={all_results[target]['R2']:.3f}")

print("\n=== Summary ===")
print(pd.DataFrame(all_results).T.to_string())